In [100]:
import pandas as pd

# Load the raw dataset
df = pd.read_csv("../data/raw/nyc311_raw.csv", low_memory=False)

# Basic shape check
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

Shape: (642796, 44)

Column names:
['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'descriptor_2', 'location_type', 'incident_zip', 'incident_address', 'street_name', 'cross_street_1', 'cross_street_2', 'intersection_street_1', 'intersection_street_2', 'address_type', 'city', 'landmark', 'facility_type', 'status', 'due_date', 'resolution_description', 'resolution_action_updated_date', 'community_board', 'council_district', 'police_precinct', 'bbl', 'borough', 'x_coordinate_state_plane', 'y_coordinate_state_plane', 'open_data_channel_type', 'park_facility_name', 'park_borough', 'vehicle_type', 'taxi_company_borough', 'taxi_pick_up_location', 'bridge_highway_name', 'bridge_highway_direction', 'road_ramp', 'bridge_highway_segment', 'latitude', 'longitude', 'location']


In [101]:
# Check null percentage per column — helps confirm which columns are mostly empty
null_pct = df.isnull().mean().sort_values(ascending=False) * 100
print(null_pct.round(1))

taxi_company_borough              99.9
road_ramp                         99.8
bridge_highway_direction          99.7
facility_type                     99.7
bridge_highway_segment            99.4
bridge_highway_name               99.4
due_date                          99.3
taxi_pick_up_location             99.0
vehicle_type                      96.4
descriptor_2                      55.0
landmark                          45.7
intersection_street_1             40.2
intersection_street_2             40.2
cross_street_1                    35.3
cross_street_2                    35.3
location_type                     12.2
bbl                                8.5
city                               3.4
street_name                        3.1
incident_address                   3.1
descriptor                         2.5
closed_date                        1.7
resolution_description             1.6
council_district                   1.5
location                           1.0
latitude                 

In [102]:
print("Number of unique complaint types:", df['complaint_type'].nunique())
print("\nTop 40 complaint types by count:")
print(df['complaint_type'].value_counts().head(40))

Number of unique complaint types: 179

Top 40 complaint types by count:
complaint_type
HEAT/HOT WATER                   103935
Noise - Residential               91350
Illegal Parking                   84494
Blocked Driveway                  32013
UNSANITARY CONDITION              19677
Noise - Street/Sidewalk           15839
Lead                              13723
PLUMBING                          12337
Water System                      10947
Abandoned Vehicle                 10489
Noise - Commercial                10458
PAINT/PLASTER                     10237
Street Condition                   9356
Noise                              9196
DOOR/WINDOW                        8345
Dirty Condition                    8318
Traffic Signal Condition           7735
Encampment                         7284
Derelict Vehicles                  7058
WATER LEAK                         6750
Missed Collection                  6564
Noise - Vehicle                    5806
Illegal Dumping                  

In [103]:
# Module mapping based on agency + complaint_type keywords
module_map = {
    "Sanitation": {
        "agency": ["DSNY"],
        "keywords": ["dirty", "sanitation", "garbage", "recycling", "missed collection", "litter"]
    },
    "Water": {
        "agency": ["DEP"],
        "keywords": ["water", "sewer", "leak", "hydrant"]
    },
    "Electricity": {
        "agency": ["DOT", "CON EDISON"],
        "keywords": ["street light", "lamp", "electric", "power"]
    },
    "Roads": {
        "agency": ["DOT"],
        "keywords": ["street condition", "pothole", "sidewalk", "road", "highway", "curb"]
    },
    "Parks": {
        "agency": ["DPR"],
        "keywords": ["park", "tree", "playground"]
    },
    "Environment": {
        "agency": ["DEP", "DOHMH"],
        "keywords": ["air quality", "noise", "asbestos", "pollution", "hazardous"]
    }
}

def assign_module(row):
    text = str(row['complaint_type']).lower()
    agency = str(row['agency']).upper()
    for module, rule in module_map.items():
        if agency in rule["agency"] and any(k in text for k in rule["keywords"]):
            return module
    return None  # doesn't belong to any of our 6 modules

df['module'] = df.apply(assign_module, axis=1)

print(df['module'].value_counts(dropna=False))

module
NaN            570324
Sanitation      16189
Water           15507
Roads           14918
Environment     12497
Parks            7894
Electricity      5467
Name: count, dtype: int64


In [104]:
# Columns to keep (from 2.3 analysis)
keep_cols = [
    'unique_key', 'created_date', 'closed_date', 'agency',
    'complaint_type', 'descriptor', 'location_type', 'incident_zip',
    'status', 'borough', 'latitude', 'longitude',
    'open_data_channel_type', 'module'
]

# Filter: only rows that matched one of our 6 modules
df_filtered = df[df['module'].notna()][keep_cols].copy()

print("Filtered shape:", df_filtered.shape)
print("\nModule distribution:")
print(df_filtered['module'].value_counts())

# Save to processed folder
df_filtered.to_csv("../data/processed/nyc311_filtered.csv", index=False)

Filtered shape: (72472, 14)

Module distribution:
module
Sanitation     16189
Water          15507
Roads          14918
Environment    12497
Parks           7894
Electricity     5467
Name: count, dtype: int64


In [105]:
print(df_filtered.shape)
df_filtered.head()

(72472, 14)


,unique_key,created_date,closed_date,agency,complaint_type,descriptor,location_type,incident_zip,status,borough,latitude,longitude,open_data_channel_type,module
35,63572271,2024-12-31T23:56:00.000,2025-01-02T19:38:00.000,DEP,Noise,Noise: air condition/ventilation equipment (NV1),NaN,10023,Closed,MANHATTAN,40.779622,-73.975988,ONLINE,Environment
62,63578403,2024-12-31T23:53:00.000,2025-01-01T09:15:00.000,DEP,Sewer,Sewer Backup (Use Comments) (SA),NaN,11234,Closed,BROOKLYN,40.625472,-73.918895,PHONE,Water
85,63582757,2024-12-31T23:50:55.000,2024-12-31T23:56:13.000,DPR,Animal in a Park,Dog Off Leash,Park,10314,Closed,STATEN ISLAND,40.599974,-74.162847,ONLINE,Parks
96,63580613,2024-12-31T23:50:00.000,2025-01-02T13:33:00.000,DOT,Street Light Condition,Street Light Out,NaN,11417,Closed,QUEENS,40.678583,-73.842153,UNKNOWN,Electricity
118,63578347,2024-12-31T23:47:27.000,2025-01-02T12:55:35.000,DSNY,Dirty Condition,Trash,Street,11421,Closed,QUEENS,40.692236,-73.865859,ONLINE,Sanitation


In [106]:
print(df_filtered.shape)

print(df_filtered["module"].value_counts())

df_filtered.head()

(72472, 14)
module
Sanitation     16189
Water          15507
Roads          14918
Environment    12497
Parks           7894
Electricity     5467
Name: count, dtype: int64


,unique_key,created_date,closed_date,agency,complaint_type,descriptor,location_type,incident_zip,status,borough,latitude,longitude,open_data_channel_type,module
35,63572271,2024-12-31T23:56:00.000,2025-01-02T19:38:00.000,DEP,Noise,Noise: air condition/ventilation equipment (NV1),NaN,10023,Closed,MANHATTAN,40.779622,-73.975988,ONLINE,Environment
62,63578403,2024-12-31T23:53:00.000,2025-01-01T09:15:00.000,DEP,Sewer,Sewer Backup (Use Comments) (SA),NaN,11234,Closed,BROOKLYN,40.625472,-73.918895,PHONE,Water
85,63582757,2024-12-31T23:50:55.000,2024-12-31T23:56:13.000,DPR,Animal in a Park,Dog Off Leash,Park,10314,Closed,STATEN ISLAND,40.599974,-74.162847,ONLINE,Parks
96,63580613,2024-12-31T23:50:00.000,2025-01-02T13:33:00.000,DOT,Street Light Condition,Street Light Out,NaN,11417,Closed,QUEENS,40.678583,-73.842153,UNKNOWN,Electricity
118,63578347,2024-12-31T23:47:27.000,2025-01-02T12:55:35.000,DSNY,Dirty Condition,Trash,Street,11421,Closed,QUEENS,40.692236,-73.865859,ONLINE,Sanitation


In [107]:
import os

print(os.getcwd())

c:\Users\bhumika nagar\Desktop\SmartCity\ml-service\notebooks


In [108]:
print(os.path.exists("data/processed/nyc311_filtered.csv"))
print(os.path.exists("../data/processed/nyc311_filtered.csv"))

False
True


In [109]:
import pandas as pd

df = pd.read_csv("../data/processed/nyc311_filtered.csv", low_memory=False)
print(df.shape)
df.info()

(72472, 14)
<class 'pandas.DataFrame'>
RangeIndex: 72472 entries, 0 to 72471
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   unique_key              72472 non-null  int64  
 1   created_date            72472 non-null  str    
 2   closed_date             70690 non-null  str    
 3   agency                  72472 non-null  str    
 4   complaint_type          72472 non-null  str    
 5   descriptor              72382 non-null  str    
 6   location_type           34808 non-null  str    
 7   incident_zip            71734 non-null  float64
 8   status                  72472 non-null  str    
 9   borough                 72472 non-null  str    
 10  latitude                69502 non-null  float64
 11  longitude               69502 non-null  float64
 12  open_data_channel_type  72472 non-null  str    
 13  module                  72472 non-null  str    
dtypes: float64(3), int64(1), str(10)
memo

In [110]:
import pandas as pd

PROCESSED_DATA = "../data/processed/nyc311_filtered.csv"

df = pd.read_csv(PROCESSED_DATA)

print(df.shape)
print(df["module"].value_counts())
print(df.isnull().mean().sort_values(ascending=False))

(72472, 14)
module
Sanitation     16189
Water          15507
Roads          14918
Environment    12497
Parks           7894
Electricity     5467
Name: count, dtype: int64
location_type             0.519704
latitude                  0.040981
longitude                 0.040981
closed_date               0.024589
incident_zip              0.010183
descriptor                0.001242
complaint_type            0.000000
agency                    0.000000
unique_key                0.000000
created_date              0.000000
borough                   0.000000
status                    0.000000
open_data_channel_type    0.000000
module                    0.000000
dtype: float64


In [111]:
import pandas as pd
raw = pd.read_csv("../data/raw/nyc311_raw.csv", low_memory=False)
print(raw.shape)
raw.head()

(642796, 44)


,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,...,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
0,63573950,2024-12-31T23:59:38.000,2025-01-01T00:26:35.000,NYPD,New York City Police Department,Illegal Fireworks,NaN,NaN,Street/Sidewalk,11218,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.640915,-73.973642,POINT (-73.973642163064 40.640914779777)
1,63574642,2024-12-31T23:59:33.000,2025-01-02T17:08:17.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,10466,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.891872,-73.860168,POINT (-73.860168452965 40.891872416493)
2,63581093,2024-12-31T23:59:32.000,2025-01-01T00:18:51.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,11221,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.688335,-73.930144,POINT (-73.930144420975 40.688334599491)
3,63574822,2024-12-31T23:59:31.000,2025-01-01T09:01:36.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,10466,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.891872,-73.860168,POINT (-73.860168452965 40.891872416493)
4,63580924,2024-12-31T23:59:21.000,2025-01-01T00:42:47.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,11230,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.634745,-73.964936,POINT (-73.96493632865 40.634744918769)


In [112]:
print(df_filtered.shape)
print(df_filtered['module'].value_counts())


(72472, 14)
module
Sanitation     16189
Water          15507
Roads          14918
Environment    12497
Parks           7894
Electricity     5467
Name: count, dtype: int64


In [113]:
df_filtered.to_csv("../data/processed/nyc311_filtered.csv", index=False)

In [114]:
import pandas as pd

df = pd.read_csv("../data/processed/nyc311_filtered.csv", low_memory=False)

print(df.shape)
df.info()

(72472, 14)
<class 'pandas.DataFrame'>
RangeIndex: 72472 entries, 0 to 72471
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   unique_key              72472 non-null  int64  
 1   created_date            72472 non-null  str    
 2   closed_date             70690 non-null  str    
 3   agency                  72472 non-null  str    
 4   complaint_type          72472 non-null  str    
 5   descriptor              72382 non-null  str    
 6   location_type           34808 non-null  str    
 7   incident_zip            71734 non-null  float64
 8   status                  72472 non-null  str    
 9   borough                 72472 non-null  str    
 10  latitude                69502 non-null  float64
 11  longitude               69502 non-null  float64
 12  open_data_channel_type  72472 non-null  str    
 13  module                  72472 non-null  str    
dtypes: float64(3), int64(1), str(10)
memo

In [115]:
print("Total rows:", len(df))
print("Duplicate unique_key rows:", df['unique_key'].duplicated().sum())

df = df.drop_duplicates(subset='unique_key', keep='first')

print("Rows after dedup:", len(df))

Total rows: 72472
Duplicate unique_key rows: 0
Rows after dedup: 72472


In [116]:
null_summary = df.isnull().sum().sort_values(ascending=False)
print(null_summary[null_summary > 0])


location_type    37664
latitude          2970
longitude         2970
closed_date       1782
incident_zip       738
descriptor          90
dtype: int64


In [117]:
# Fill categorical fields
df['descriptor'] = df['descriptor'].fillna('Unknown')
df['location_type'] = df['location_type'].fillna('Unspecified')

# Drop rows missing critical geographic fields
before = len(df)
df = df.dropna(subset=['incident_zip', 'latitude', 'longitude'])
after = len(df)

print(f"Dropped {before - after} rows missing zip/lat/long")
print(f"Remaining rows: {after}")

Dropped 2973 rows missing zip/lat/long
Remaining rows: 69499


In [118]:
df['created_date'] = pd.to_datetime(df['created_date'], errors='coerce')
df['closed_date'] = pd.to_datetime(df['closed_date'], errors='coerce')

print(df[['created_date', 'closed_date']].dtypes)
print(df[['created_date', 'closed_date']].head())

created_date    datetime64[us]
closed_date     datetime64[us]
dtype: object
         created_date         closed_date
0 2024-12-31 23:56:00 2025-01-02 19:38:00
1 2024-12-31 23:53:00 2025-01-01 09:15:00
2 2024-12-31 23:50:55 2024-12-31 23:56:13
3 2024-12-31 23:50:00 2025-01-02 13:33:00
4 2024-12-31 23:47:27 2025-01-02 12:55:35


In [119]:
# Check the actual range in your data
print(df[['latitude', 'longitude']].describe())

           latitude     longitude
count  69499.000000  69499.000000
mean      40.715665    -73.931242
std        0.082581      0.093815
min       40.499464    -74.254937
25%       40.656205    -73.980598
50%       40.714904    -73.939822
75%       40.767515    -73.873191
max       40.912869    -73.700597


In [120]:
before = len(df)

df = df[
    (df['latitude'].between(40.4, 40.95)) &
    (df['longitude'].between(-74.3, -73.65))
]

after = len(df)
print(f"Dropped {before - after} rows with invalid coordinates")
print(f"Remaining rows: {after}")

Dropped 0 rows with invalid coordinates
Remaining rows: 69499


In [121]:
df = df.rename(columns={
    'incident_zip': 'zip_code',
    'open_data_channel_type': 'submission_channel'
})

print(df.columns.tolist())

['unique_key', 'created_date', 'closed_date', 'agency', 'complaint_type', 'descriptor', 'location_type', 'zip_code', 'status', 'borough', 'latitude', 'longitude', 'submission_channel', 'module']


In [122]:
# Check current unique values
print(df['borough'].unique())
print(df['status'].unique())

<StringArray>
['MANHATTAN', 'BROOKLYN', 'STATEN ISLAND', 'QUEENS', 'BRONX', 'Unspecified']
Length: 6, dtype: str
<StringArray>
['Closed', 'Open', 'Started', 'In Progress', 'Assigned', 'Pending']
Length: 6, dtype: str


In [123]:
# Standardize casing and strip whitespace
df['borough'] = df['borough'].str.strip().str.title()
df['status'] = df['status'].str.strip().str.title()
df['location_type'] = df['location_type'].str.strip().str.title()

print(df['borough'].value_counts())

borough
Brooklyn         22248
Queens           19317
Manhattan        13549
Bronx             8729
Staten Island     5650
Unspecified          6
Name: count, dtype: int64


In [124]:
print("Final shape:", df.shape)
print("\nRemaining nulls:")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)
print("\nBorough distribution:")
print(df['borough'].value_counts())

Final shape: (69499, 14)

Remaining nulls:
unique_key               0
created_date             0
closed_date           1664
agency                   0
complaint_type           0
descriptor               0
location_type            0
zip_code                 0
status                   0
borough                  0
latitude                 0
longitude                0
submission_channel       0
module                   0
dtype: int64

Data types:
unique_key                     int64
created_date          datetime64[us]
closed_date           datetime64[us]
agency                           str
complaint_type                   str
descriptor                       str
location_type                    str
zip_code                     float64
status                           str
borough                          str
latitude                     float64
longitude                    float64
submission_channel               str
module                           str
dtype: object

Borough distribution

In [125]:
df.to_csv("../data/processed/nyc311_cleaned.csv", index=False)
print("Saved cleaned dataset:", df.shape)

Saved cleaned dataset: (69499, 14)


In [126]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("../data/processed/nyc311_cleaned.csv", parse_dates=['created_date', 'closed_date'])

print(df.shape)
df.head()

(69499, 14)


,unique_key,created_date,closed_date,agency,complaint_type,descriptor,location_type,zip_code,status,borough,latitude,longitude,submission_channel,module
0,63572271,2024-12-31 23:56:00,2025-01-02 19:38:00,DEP,Noise,Noise: air condition/ventilation equipment (NV1),Unspecified,10023.0,Closed,Manhattan,40.779622,-73.975988,ONLINE,Environment
1,63578403,2024-12-31 23:53:00,2025-01-01 09:15:00,DEP,Sewer,Sewer Backup (Use Comments) (SA),Unspecified,11234.0,Closed,Brooklyn,40.625472,-73.918895,PHONE,Water
2,63582757,2024-12-31 23:50:55,2024-12-31 23:56:13,DPR,Animal in a Park,Dog Off Leash,Park,10314.0,Closed,Staten Island,40.599974,-74.162847,ONLINE,Parks
3,63580613,2024-12-31 23:50:00,2025-01-02 13:33:00,DOT,Street Light Condition,Street Light Out,Unspecified,11417.0,Closed,Queens,40.678583,-73.842153,UNKNOWN,Electricity
4,63578347,2024-12-31 23:47:27,2025-01-02 12:55:35,DSNY,Dirty Condition,Trash,Street,11421.0,Closed,Queens,40.692236,-73.865859,ONLINE,Sanitation


In [127]:
print(df.dtypes[['created_date', 'closed_date']])

created_date    datetime64[us]
closed_date     datetime64[us]
dtype: object


In [128]:
module_counts = df['module'].value_counts().reset_index()
module_counts.columns = ['module', 'count']

fig = px.bar(
    module_counts,
    x='module', y='count',
    title='Complaint Distribution by Module',
    color='module',
    text='count'
)
fig.update_layout(showlegend=False)
fig.show()

**Insight:** Sanitation (16,096) and Water (15,331) are the two largest categories, followed by Roads and Environment. Electricity is the smallest module (5,166) — this is expected, since NYC 311 does not route power outages through a dedicated agency; most electrical complaints are captured indirectly through DOT streetlight reports. This imbalance should be considered when training classification models in Phase 6.

In [129]:
borough_counts = df['borough'].value_counts().reset_index()
borough_counts.columns = ['borough', 'count']

fig = px.bar(
    borough_counts,
    x='borough', y='count',
    title='Complaints by Borough',
    color='borough',
    text='count'
)
fig.update_layout(showlegend=False)
fig.show()

**Insight:** Complaint volume roughly tracks real NYC population ranking — Brooklyn (22,248) and Queens (19,317) lead, followed by Manhattan, Bronx, and Staten Island. This consistency confirms the dataset is not skewed by a filtering or scraping error. A small "Unspecified" borough category (6 rows) exists and is treated as negligible noise.

In [130]:
df['date_only'] = df['created_date'].dt.date
daily_counts = df.groupby('date_only').size().reset_index(name='count')

fig = px.line(
    daily_counts,
    x='date_only', y='count',
    title='Daily Complaint Volume Over Time'
)
fig.show()

**Insight:** Complaint volume shows a clear repeating weekly cycle — regular peaks and dips roughly every 7 days — most likely reflecting lower complaint activity on weekends. This supports adding a `day_of_week` / `is_weekend` feature in Phase 5.

In [131]:
df['year_month'] = df['created_date'].dt.to_period('M').astype(str)
monthly = df.groupby(['year_month', 'module']).size().reset_index(name='count')

fig = px.line(
    monthly,
    x='year_month', y='count', color='module',
    title='Monthly Complaint Trends by Module',
    markers=True
)
fig.show()

**Insight:** Water and Electricity complaints show the steepest upward trend across the observed months, while Environment and Roads grow more gradually. Note: this dataset spans roughly Oct 2024–Jan 2025 (a few months), not a full multi-year range, so seasonal patterns (e.g., summer vs. winter) cannot be fully assessed from this extract alone.

In [132]:
# Only use resolved complaints (closed_date not null)
resolved = df.dropna(subset=['closed_date']).copy()
resolved['response_hours'] = (resolved['closed_date'] - resolved['created_date']).dt.total_seconds() / 3600

# Filter out negative/absurd values (data entry errors)
resolved = resolved[(resolved['response_hours'] >= 0) & (resolved['response_hours'] <= 24*90)]  # cap at 90 days

agency_perf = resolved.groupby('agency')['response_hours'].median().sort_values().reset_index()

fig = px.bar(
    agency_perf,
    x='agency', y='response_hours',
    title='Median Response Time by Agency (Hours)'
)
fig.show()

**Insight:** DOHMH resolves complaints fastest (~12 hours median), while DOT and DPR are slowest (~65+ hours). This makes sense — health-related complaints (DOHMH) are often quick administrative responses, while road (DOT) and park (DPR) issues typically require physical repair work.

In [133]:
fig = px.histogram(
    resolved,
    x='response_hours',
    nbins=100,
    title='Distribution of Response Times (Hours)'
)
fig.show()

Response time is heavily right-skewed — most complaints resolve within a few days, but a long tail extends to 60-90+ days. This will require a log-transformation of the target variable for response-time prediction models in Phase 6.

In [134]:
df = df[
    (df['latitude'].between(40.49, 40.92)) &
    (df['longitude'].between(-74.26, -73.68))
]
print(df.shape)

(69499, 16)


In [135]:
df.to_csv("../data/processed/nyc311_cleaned.csv", index=False)

In [136]:
print("Before:", df.shape)

# Filter 1: keep only valid NYC ZIP codes (already applied, kept for clarity)
nyc_zip_ranges = [
    (10001, 10282), (10301, 10314), (10451, 10475),
    (11004, 11109), (11201, 11256), (11351, 11697),
]
df['zip_code'] = df['zip_code'].astype(int)
df = df[df['zip_code'].apply(lambda z: any(low <= z <= high for low, high in nyc_zip_ranges))]

# Filter 2: keep only coordinates within NYC's actual bounding box
df = df[
    (df['latitude'].between(40.49, 40.92)) &
    (df['longitude'].between(-74.26, -73.68))
]

print("After:", df.shape)

df.to_csv("../data/processed/nyc311_cleaned.csv", index=False)
print("Saved.")

Before: (69499, 16)
After: (69440, 16)
Saved.


In [137]:
fig = px.scatter_mapbox(
    sample,
    lat='latitude', lon='longitude',
    color='borough',
    hover_data=['complaint_type', 'module'],
    zoom=9,
    height=700,
    title='Complaint Hotspots by Borough'
)
fig.update_layout(mapbox_style="open-street-map")
fig.show()

C:\Users\bhumika nagar\AppData\Local\Temp\ipykernel_8800\2972292771.py:1: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


In [138]:
cross = df.groupby(['borough', 'module']).size().reset_index(name='count')

fig = px.bar(
    cross,
    x='borough', y='count', color='module',
    title='Complaints by Borough and Module',
    barmode='stack'
)
fig.show()

Manhattan shows a disproportionately high share of Environment (noise) complaints relative to its total volume, likely reflecting its density and nightlife activity. Staten Island shows a comparatively higher share of Water complaints, possibly linked to older infrastructur

## Phase 4 Summary

Key takeaways carried into Phase 5 (Feature Engineering):
- Weekly cyclical pattern in complaint volume → build `day_of_week` and `is_weekend` features
- Response time is right-skewed → apply log-transform before modeling
- Module distribution is imbalanced (Electricity is smallest) → consider this when evaluating classification model performance
- Borough and module patterns differ meaningfully → borough and module are both strong candidate features
- Geographic clustering is visually distinct by borough → borough itself may be sufficient as a geographic feature, reducing the need for complex geo-clustering

In [139]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/nyc311_cleaned.csv", parse_dates=['created_date', 'closed_date'])
print(df.shape)
df.head()

(69440, 16)


,unique_key,created_date,closed_date,agency,complaint_type,descriptor,location_type,zip_code,status,borough,latitude,longitude,submission_channel,module,date_only,year_month
0,63572271,2024-12-31 23:56:00,2025-01-02 19:38:00,DEP,Noise,Noise: air condition/ventilation equipment (NV1),Unspecified,10023,Closed,Manhattan,40.779622,-73.975988,ONLINE,Environment,2024-12-31,2024-12
1,63578403,2024-12-31 23:53:00,2025-01-01 09:15:00,DEP,Sewer,Sewer Backup (Use Comments) (SA),Unspecified,11234,Closed,Brooklyn,40.625472,-73.918895,PHONE,Water,2024-12-31,2024-12
2,63582757,2024-12-31 23:50:55,2024-12-31 23:56:13,DPR,Animal in a Park,Dog Off Leash,Park,10314,Closed,Staten Island,40.599974,-74.162847,ONLINE,Parks,2024-12-31,2024-12
3,63580613,2024-12-31 23:50:00,2025-01-02 13:33:00,DOT,Street Light Condition,Street Light Out,Unspecified,11417,Closed,Queens,40.678583,-73.842153,UNKNOWN,Electricity,2024-12-31,2024-12
4,63578347,2024-12-31 23:47:27,2025-01-02 12:55:35,DSNY,Dirty Condition,Trash,Street,11421,Closed,Queens,40.692236,-73.865859,ONLINE,Sanitation,2024-12-31,2024-12


In [140]:
print(df.dtypes[['created_date', 'closed_date']])

created_date    datetime64[us]
closed_date     datetime64[us]
dtype: object


In [141]:
df['response_hours'] = (df['closed_date'] - df['created_date']).dt.total_seconds() / 3600

print(df['response_hours'].describe())

count    67781.000000
mean       305.941778
std       1133.575313
min      -1584.016667
25%         15.370278
50%         43.356389
75%        143.771389
max      14989.716667
Name: response_hours, dtype: float64


In [142]:
invalid_response = (df['response_hours'] < 0).sum()
print(f"Rows with negative response time: {invalid_response}")

# Negative response time is a data error (closed before created) — remove these
df = df[(df['response_hours'] >= 0) | (df['response_hours'].isna())]
print(df.shape)

Rows with negative response time: 119
(69321, 17)


In [143]:
# Cap at 90 days, consistent with EDA
df['response_hours_capped'] = df['response_hours'].clip(upper=90*24)

In [144]:
df['response_hours_log'] = np.log1p(df['response_hours_capped'])

print(df[['response_hours', 'response_hours_capped', 'response_hours_log']].describe())

       response_hours  response_hours_capped  response_hours_log
count    67662.000000           67662.000000        67662.000000
mean       306.722342             189.285360            3.721052
std       1134.360040             446.862007            1.825381
min          0.000000               0.000000            0.000000
25%         15.516667              15.516667            2.804370
50%         43.508333              43.508333            3.795676
75%        143.929167             143.929167            4.976245
max      14989.716667            2160.000000            7.678326


In [145]:
import plotly.express as px
fig = px.histogram(df, x='response_hours_log', nbins=50, title='Log-Transformed Response Time')
fig.show()

In [146]:
df['is_resolved'] = df['closed_date'].notna().astype(int)

print(df['is_resolved'].value_counts())

is_resolved
1    67662
0     1659
Name: count, dtype: int64


In [147]:
df['day_of_week'] = df['created_date'].dt.day_name()
df['day_of_week_num'] = df['created_date'].dt.dayofweek  # 0=Monday, 6=Sunday

print(df['day_of_week'].value_counts())

day_of_week
Tuesday      12306
Monday       11141
Wednesday    11126
Thursday     10324
Friday       10173
Sunday        7438
Saturday      6813
Name: count, dtype: int64


In [148]:
df['is_weekend'] = df['day_of_week_num'].isin([5, 6]).astype(int)

print(df['is_weekend'].value_counts())

is_weekend
0    55070
1    14251
Name: count, dtype: int64


In [149]:
df['hour_of_day'] = df['created_date'].dt.hour

print(df['hour_of_day'].value_counts().sort_index())

hour_of_day
0      971
1      618
2      427
3      305
4      371
5      545
6     1556
7     2960
8     5459
9     5733
10    5634
11    5138
12    4721
13    5006
14    5229
15    4548
16    4061
17    3172
18    2757
19    2576
20    2097
21    2018
22    1881
23    1538
Name: count, dtype: int64


In [150]:
fig = px.histogram(df, x='hour_of_day', nbins=24, title='Complaints by Hour of Day')
fig.show()

In [151]:
df['month'] = df['created_date'].dt.month

def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df['season'] = df['month'].apply(get_season)

print(df['season'].value_counts())

season
Fall      37400
Winter    31921
Name: count, dtype: int64


In [152]:
high_priority_keywords = ['emergency', 'gas', 'fire', 'water main', 'structural', 'collapse']
medium_priority_keywords = ['noise', 'illegal', 'blocked', 'leak']

def assign_priority(complaint_type):
    text = str(complaint_type).lower()
    if any(k in text for k in high_priority_keywords):
        return 'High'
    elif any(k in text for k in medium_priority_keywords):
        return 'Medium'
    else:
        return 'Low'

df['priority'] = df['complaint_type'].apply(assign_priority)

print(df['priority'].value_counts())

priority
Low       59665
Medium     9656
Name: count, dtype: int64


In [153]:
print(df['module'].value_counts())

module
Sanitation     16083
Water          15327
Roads          12657
Environment    12376
Parks           7849
Electricity     5029
Name: count, dtype: int64


In [154]:
# One-hot encode low-cardinality categoricals
categorical_cols = ['borough', 'module', 'priority', 'season', 'submission_channel']

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(df_encoded.shape)
print(df_encoded.columns.tolist())

(69321, 38)
['unique_key', 'created_date', 'closed_date', 'agency', 'complaint_type', 'descriptor', 'location_type', 'zip_code', 'status', 'latitude', 'longitude', 'date_only', 'year_month', 'response_hours', 'response_hours_capped', 'response_hours_log', 'is_resolved', 'day_of_week', 'day_of_week_num', 'is_weekend', 'hour_of_day', 'month', 'borough_Brooklyn', 'borough_Manhattan', 'borough_Queens', 'borough_Staten Island', 'borough_Unspecified', 'module_Environment', 'module_Parks', 'module_Roads', 'module_Sanitation', 'module_Water', 'priority_Medium', 'season_Winter', 'submission_channel_ONLINE', 'submission_channel_OTHER', 'submission_channel_PHONE', 'submission_channel_UNKNOWN']


In [155]:
final_features = [
    'unique_key', 'created_date', 'closed_date', 'complaint_type',
    'response_hours', 'response_hours_capped', 'response_hours_log',
    'is_resolved', 'day_of_week', 'day_of_week_num', 'is_weekend',
    'hour_of_day', 'month', 'season', 'priority', 'module', 'borough',
    'latitude', 'longitude', 'zip_code', 'status', 'agency'
]

df_final = df[final_features].copy()

print(df_final.shape)
df_final.head()

df_final.to_csv("../data/processed/nyc311_features.csv", index=False)
print("Saved feature-engineered dataset.")

(69321, 22)
Saved feature-engineered dataset.


In [156]:
print(df_final.isnull().sum())

unique_key                  0
created_date                0
closed_date              1659
complaint_type              0
response_hours           1659
response_hours_capped    1659
response_hours_log       1659
is_resolved                 0
day_of_week                 0
day_of_week_num             0
is_weekend                  0
hour_of_day                 0
month                       0
season                      0
priority                    0
module                      0
borough                     0
latitude                    0
longitude                   0
zip_code                    0
status                      0
agency                      0
dtype: int64


In [158]:
import pandas as pd
df = pd.read_csv("../data/processed/nyc311_features.csv")
print(df.shape)
print(df.columns.tolist())
print(df.isnull().sum())

(69321, 22)
['unique_key', 'created_date', 'closed_date', 'complaint_type', 'response_hours', 'response_hours_capped', 'response_hours_log', 'is_resolved', 'day_of_week', 'day_of_week_num', 'is_weekend', 'hour_of_day', 'month', 'season', 'priority', 'module', 'borough', 'latitude', 'longitude', 'zip_code', 'status', 'agency']
unique_key                  0
created_date                0
closed_date              1659
complaint_type              0
response_hours           1659
response_hours_capped    1659
response_hours_log       1659
is_resolved                 0
day_of_week                 0
day_of_week_num             0
is_weekend                  0
hour_of_day                 0
month                       0
season                      0
priority                    0
module                      0
borough                     0
latitude                    0
longitude                   0
zip_code                    0
status                      0
agency                      0
dtype: int64